In [1]:
# calculator_app.py
import math
import sys
from typing import Callable, List, Tuple, Optional

# Optional dependencies for graphing
try:
    import numpy as np
    import matplotlib.pyplot as plt
    HAS_MPL = True
except Exception:
    HAS_MPL = False

# -------------------------
# Multi-function calculator
# -------------------------
def add(a: float, b: float) -> float:
    return a + b

def subtract(a: float, b: float) -> float:
    return a - b

def multiply(a: float, b: float) -> float:
    return a * b

def divide(a: float, b: float) -> float:
    if b == 0:
        raise ZeroDivisionError("Division by zero")
    return a / b

def power(a: float, b: float) -> float:
    return a ** b

def sqrt(a: float) -> float:
    if a < 0:
        raise ValueError("sqrt of negative number")
    return math.sqrt(a)

def factorial(n: int) -> int:
    if n < 0:
        raise ValueError("factorial of negative")
    return math.factorial(n)

def comb(n: int, k: int) -> int:
    return math.comb(n, k)

def perm(n: int, k: int) -> int:
    return math.perm(n, k)

def sin(x: float, deg: bool=False) -> float:
    rad = math.radians(x) if deg else x
    return math.sin(rad)

def cos(x: float, deg: bool=False) -> float:
    rad = math.radians(x) if deg else x
    return math.cos(rad)

def tan(x: float, deg: bool=False) -> float:
    rad = math.radians(x) if deg else x
    return math.tan(rad)

def ln(x: float) -> float:
    return math.log(x)

def log10(x: float) -> float:
    return math.log10(x)

# -------------------------
# Graphing calculator
# -------------------------
def parse_expression(expr: str) -> Callable[[float], float]:
    """
    Safely convert a single-variable expression in 'x' to a callable f(x).
    Allowed names: math functions and constants, numpy functions if available.
    """
    allowed_names = {k: getattr(math, k) for k in dir(math) if not k.startswith("__")}
    # add builtins we want
    allowed_names.update({
        'sqrt': math.sqrt, 'pi': math.pi, 'e': math.e,
        'abs': abs, 'pow': pow
    })
    if HAS_MPL:
        # allow numpy for vectorized ops (sin, cos, exp, etc.)
        import numpy as _np
        np_funcs = {k: getattr(_np, k) for k in ['sin','cos','tan','exp','log','log10','sqrt','abs','power']}
        allowed_names.update(np_funcs)
        allowed_names['np'] = _np

    def f(x):
        local = {'x': x}
        return eval(expr, {"__builtins__": {}}, {**allowed_names, **local})
    return f

def plot_function(expr: str, x_min: float=-10, x_max: float=10, points: int=1000, show: bool=True) -> Optional[Tuple[np.ndarray, np.ndarray]]:
    """
    Plots expression in variable x. Returns (xs, ys) if plotting succeeded.
    """
    if not HAS_MPL:
        raise RuntimeError("matplotlib/numpy required for plotting")
    f = parse_expression(expr)
    xs = np.linspace(x_min, x_max, points)
    # evaluate safely elementwise if numpy available; otherwise map
    try:
        ys = f(xs)
    except Exception:
        # fallback: evaluate elementwise
        ys = np.array([f(float(xi)) for xi in xs], dtype=float)
    plt.figure(figsize=(8,4))
    plt.plot(xs, ys, label=expr)
    plt.axhline(0, color='black', linewidth=0.8)
    plt.axvline(0, color='black', linewidth=0.8)
    plt.title(f"y = {expr}")
    plt.xlabel("x")
    plt.ylabel("y")
    plt.grid(True)
    plt.legend()
    if show:
        plt.show()
    return xs, ys

# -------------------------
# Financial calculator
# -------------------------
def loan_payment(principal: float, annual_rate: float, years: float, payments_per_year: int=12) -> float:
    """
    Returns periodic payment amount given principal, annual interest rate (as decimal), years, and payments per year.
    """
    r = annual_rate / payments_per_year
    n = int(years * payments_per_year)
    if r == 0:
        return principal / n
    payment = principal * (r * (1 + r)**n) / ((1 + r)**n - 1)
    return payment

def amortization_schedule(principal: float, annual_rate: float, years: float, payments_per_year: int=12) -> List[dict]:
    """
    Returns list of amortization entries: payment_no, payment, principal_paid, interest_paid, remaining_balance.
    """
    payment = loan_payment(principal, annual_rate, years, payments_per_year)
    r = annual_rate / payments_per_year
    n = int(years * payments_per_year)
    balance = principal
    schedule = []
    for i in range(1, n+1):
        interest = balance * r
        principal_paid = payment - interest
        balance = max(0.0, balance - principal_paid)
        schedule.append({
            'payment_no': i,
            'payment': round(payment, 10),
            'principal_paid': round(principal_paid, 10),
            'interest_paid': round(interest, 10),
            'remaining_balance': round(balance, 10)
        })
    return schedule

def future_value(present_value: float, annual_rate: float, years: float, compounds_per_year: int=1) -> float:
    r = annual_rate / compounds_per_year
    n = int(years * compounds_per_year)
    return present_value * (1 + r)**n

def present_value(future_value_amount: float, annual_rate: float, years: float, compounds_per_year: int=1) -> float:
    r = annual_rate / compounds_per_year
    n = int(years * compounds_per_year)
    return future_value_amount / (1 + r)**n

def roi_gain(initial_investment: float, final_value: float) -> float:
    return (final_value - initial_investment) / initial_investment

# -------------------------
# Command-line interface
# -------------------------
def print_menu():
    print("\nSelect calculator:")
    print("1) Multi-function calculator")
    print("2) Graphing calculator (requires matplotlib & numpy)")
    print("3) Financial calculator")
    print("0) Exit")

def run_multi():
    print("\nMulti-function calculator. Examples of operations: + - * / pow sqrt fact comb perm sin cos tan ln log10")
    expr = input("Enter expression using variables a and b when needed (examples: a+b, pow(a,b), sin(a) ). Or single value: '42' >>> ").strip()
    # naive evaluation: map a and b if present
    try:
        if 'a' in expr or 'b' in expr:
            a = float(input("a = "))
            b = float(input("b = "))
            safe_globals = {"__builtins__": {}}
            safe_locals = {'a': a, 'b': b,
                           'add': add, 'subtract': subtract, 'multiply': multiply, 'divide': divide,
                           'pow': power, 'sqrt': sqrt, 'fact': factorial, 'factorial': factorial,
                           'comb': comb, 'perm': perm,
                           'sin': sin, 'cos': cos, 'tan': tan, 'ln': ln, 'log10': log10, 'pi': math.pi, 'e': math.e}
            result = eval(expr, safe_globals, safe_locals)
        else:
            # single numeric expression
            safe_globals = {"__builtins__": {}}
            safe_locals = {'sqrt': sqrt, 'pow': power, 'sin': sin, 'cos': cos, 'tan': tan, 'ln': ln, 'log10': log10, 'pi': math.pi, 'e': math.e}
            result = eval(expr, safe_globals, safe_locals)
        print("Result:", result)
    except Exception as e:
        print("Error:", e)

def run_graph():
    if not HAS_MPL:
        print("Graphing requires numpy and matplotlib. Install them and retry.")
        return
    expr = input("Enter function in x (e.g., 'sin(x)', 'x**2 + 2*x - 3', 'np.sin(x) + x/2') >>> ").strip()
    try:
        x_min = float(input("x min (default -10): ") or -10)
        x_max = float(input("x max (default 10): ") or 10)
        points = int(input("Points (default 1000): ") or 1000)
        plot_function(expr, x_min, x_max, points, show=True)
    except Exception as e:
        print("Error plotting:", e)

def run_financial():
    print("\nFinancial calculator options:")
    print("1) Loan payment")
    print("2) Amortization schedule")
    print("3) Future value")
    print("4) Present value")
    print("5) ROI")
    opt = input("Choose option >>> ").strip()
    try:
        if opt == '1':
            principal = float(input("Principal >>> "))
            rate = float(input("Annual rate (e.g., 0.05) >>> "))
            years = float(input("Years >>> "))
            ppy = int(input("Payments per year (default 12) >>> ") or 12)
            payment = loan_payment(principal, rate, years, ppy)
            print(f"Periodic payment: {payment:.2f}")
        elif opt == '2':
            principal = float(input("Principal >>> "))
            rate = float(input("Annual rate (e.g., 0.05) >>> "))
            years = float(input("Years >>> "))
            ppy = int(input("Payments per year (default 12) >>> ") or 12)
            sched = amortization_schedule(principal, rate, years, ppy)
            print("payment_no, payment, principal_paid, interest_paid, remaining_balance")
            for row in sched:
                print(f"{row['payment_no']}, {row['payment']:.2f}, {row['principal_paid']:.2f}, {row['interest_paid']:.2f}, {row['remaining_balance']:.2f}")
        elif opt == '3':
            pv = float(input("Present value >>> "))
            rate = float(input("Annual rate (e.g., 0.05) >>> "))
            years = float(input("Years >>> "))
            cpy = int(input("Compounds per year (default 1) >>> ") or 1)
            fv = future_value(pv, rate, years, cpy)
            print(f"Future value: {fv:.2f}")
        elif opt == '4':
            fv = float(input("Future value >>> "))
            rate = float(input("Annual rate (e.g., 0.05) >>> "))
            years = float(input("Years >>> "))
            cpy = int(input("Compounds per year (default 1) >>> ") or 1)
            pv = present_value(fv, rate, years, cpy)
            print(f"Present value: {pv:.2f}")
        elif opt == '5':
            initial = float(input("Initial investment >>> "))
            final = float(input("Final value >>> "))
            r = roi_gain(initial, final)
            print(f"ROI (decimal): {r:.4f}  ROI (%): {r*100:.2f}%")
        else:
            print("Invalid option")
    except Exception as e:
        print("Error:", e)

def main():
    print("Calculator App")
    while True:
        print_menu()
        choice = input("Choice >>> ").strip()
        if choice == '1':
            run_multi()
        elif choice == '2':
            run_graph()
        elif choice == '3':
            run_financial()
        elif choice == '0':
            print("Goodbye")
            break
        else:
            print("Invalid choice")

if __name__ == "__main__":
    main()

Calculator App

Select calculator:
1) Multi-function calculator
2) Graphing calculator (requires matplotlib & numpy)
3) Financial calculator
0) Exit
Choice >>> 1

Multi-function calculator. Examples of operations: + - * / pow sqrt fact comb perm sin cos tan ln log10
Enter expression using variables a and b when needed (examples: a+b, pow(a,b), sin(a) ). Or single value: '42' >>> 5 * 5
Result: 25

Select calculator:
1) Multi-function calculator
2) Graphing calculator (requires matplotlib & numpy)
3) Financial calculator
0) Exit
Choice >>> 3

Financial calculator options:
1) Loan payment
2) Amortization schedule
3) Future value
4) Present value
5) ROI
Choose option >>> 1
Principal >>> 200
Annual rate (e.g., 0.05) >>> 10
Years >>> 5
Payments per year (default 12) >>> 12
Periodic payment: 166.67

Select calculator:
1) Multi-function calculator
2) Graphing calculator (requires matplotlib & numpy)
3) Financial calculator
0) Exit


KeyboardInterrupt: Interrupted by user